# ViT Model Testing: Metrics and Batched Predictions

This notebook evaluates trained ViT models with focus on:
1. **Quantitative Metrics**: PSNR, SSIM, NCC, MSE for all tasks
2. **Batched Inference**: Efficient batched prediction
3. **Performance Analysis**: Speed, memory usage
4. **Batch Prediction Export**: Save results for external analysis

**Use `visualize_degradations.ipynb` for visualization of degradations and encoder outputs.**

## 1. Setup

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from pathlib import Path
import sys
import time
from tqdm.auto import tqdm
from torch.utils.data import DataLoader
import json

# Add src to path
sys.path.insert(0, 'src')
sys.path.insert(0, str(Path.cwd()))

# Import
from orochi.configs.model_configs import ViT3DConfig
from vit_model import ViTULight
from finetune_with_wandb import BiomedicalDataset
import losses

# Device
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Seeds
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

print("✓ Setup complete")

## 2. Configuration and Model

In [ ]:
# Configuration
config = ViT3DConfig()

print("Config:")
print(f"  Image size: {config.img_size}")
print(f"  Batch size: {config.batch_size}")
print(f"  Task: {config.task}")
print(f"  Embed dim: {config.embed_dim}")

In [ ]:
# Create model
print("\nCreating model...")
model = ViTULight(config).to(device)
model.eval()

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params/1e6:.2f}M")
print(f"Trainable parameters: {trainable_params/1e6:.2f}M ({100*trainable_params/total_params:.1f}%)")

In [ ]:
# Load checkpoint (UPDATE PATH HERE)
CHECKPOINT_PATH = "/path/to/your/checkpoint.pth"  # <-- UPDATE THIS

if Path(CHECKPOINT_PATH).exists():
    print(f"\nLoading checkpoint: {CHECKPOINT_PATH}")
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
    
    if 'model_state_dict' in checkpoint:
        model.load_state_dict(checkpoint['model_state_dict'])
        print(f"  Epoch: {checkpoint.get('epoch', 'N/A')}")
        print(f"  Best loss: {checkpoint.get('best_loss', 'N/A'):.6f}")
    elif 'state_dict' in checkpoint:
        model.load_state_dict(checkpoint['state_dict'])
    else:
        model.load_state_dict(checkpoint)
    
    print("✓ Checkpoint loaded")
else:
    print(f"\n⚠️  Warning: Checkpoint not found at {CHECKPOINT_PATH}")
    print("Using random weights for demonstration")
    print("\nTo use a trained checkpoint:")
    print("  1. Update CHECKPOINT_PATH variable above")
    print("  2. Re-run this cell")

## 3. Dataset and DataLoader

In [ ]:
# Test batch size (can be larger for faster inference)
TEST_BATCH_SIZE = 4

# Create dataset
print("Loading dataset...")
val_dataset = BiomedicalDataset(
    data_root=config.data_root,
    datasets=['hipsc_3d', 'hipsc_2d', 'hipct_2d', 'idr_2d'],
    img_size=config.img_size,
    split='val',
    val_split=0.1,
    use_random_crop=False  # Deterministic for testing
)

val_loader = DataLoader(
    val_dataset,
    batch_size=TEST_BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
    drop_last=False
)

print(f"\nDataset: {len(val_dataset)} samples")
print(f"Batches: {len(val_loader)} (batch_size={TEST_BATCH_SIZE})")
print("✓ Data ready")

## 4. Metric Functions

In [ ]:
@torch.no_grad()
def compute_metrics(pred, target):
    """Compute all metrics between prediction and target.
    
    Args:
        pred: Predicted image (numpy or torch)
        target: Target image (numpy or torch)
    
    Returns:
        dict: Dictionary of metric values
    """
    # Convert to torch if needed
    if isinstance(pred, np.ndarray):
        pred = torch.from_numpy(pred)
    if isinstance(target, np.ndarray):
        target = torch.from_numpy(target)
    
    # MSE
    mse = torch.mean((pred - target) ** 2).item()
    
    # MAE
    mae = torch.mean(torch.abs(pred - target)).item()
    
    # PSNR
    if mse > 0:
        psnr = 20 * np.log10(1.0 / np.sqrt(mse))
    else:
        psnr = float('inf')
    
    # SSIM (simplified)
    C1 = 0.01 ** 2
    C2 = 0.03 ** 2
    mu1 = torch.mean(pred)
    mu2 = torch.mean(target)
    sigma1_sq = torch.var(pred)
    sigma2_sq = torch.var(target)
    sigma12 = torch.mean((pred - mu1) * (target - mu2))
    ssim = ((2 * mu1 * mu2 + C1) * (2 * sigma12 + C2)) / \
           ((mu1**2 + mu2**2 + C1) * (sigma1_sq + sigma2_sq + C2))
    ssim = ssim.item()
    
    # NCC
    pred_norm = (pred - pred.mean()) / (pred.std() + 1e-8)
    target_norm = (target - target.mean()) / (target.std() + 1e-8)
    ncc = torch.mean(pred_norm * target_norm).item()
    
    return {
        'mse': mse,
        'mae': mae,
        'psnr': psnr,
        'ssim': ssim,
        'ncc': ncc
    }

print("✓ Metric functions defined")

## 5. Batched Inference

In [ ]:
# Number of batches to evaluate (set to None for full dataset)
NUM_BATCHES = 50  # Adjust based on your needs

# Results storage
results = {
    'registration': {'mse': [], 'mae': [], 'psnr': [], 'ssim': [], 'ncc': []},
    'fusion': {'mse': [], 'mae': [], 'psnr': [], 'ssim': [], 'ncc': []},
    'super_resolution': {'mse': [], 'mae': [], 'psnr': [], 'ssim': [], 'ncc': []},
    'isotropic_restoration': {'mse': [], 'mae': [], 'psnr': [], 'ssim': [], 'ncc': []}
}

# Timing
inference_times = []
total_samples = 0

print(f"\nRunning batched inference on {NUM_BATCHES if NUM_BATCHES else len(val_loader)} batches...")
print(f"Batch size: {TEST_BATCH_SIZE}\n")

with torch.no_grad():
    for batch_idx, batch in enumerate(tqdm(val_loader, total=NUM_BATCHES if NUM_BATCHES else len(val_loader))):
        if NUM_BATCHES and batch_idx >= NUM_BATCHES:
            break
        
        # Move to device
        images = batch['image'].to(device)
        batch_size_actual = images.shape[0]
        
        # Time inference
        start_time = time.time()
        logits, aux_loss = model(images)
        inference_time = time.time() - start_time
        
        inference_times.append(inference_time)
        total_samples += batch_size_actual
        
        # Convert raw to torch
        raw = torch.from_numpy(logits['raw'])
        
        # Compute metrics for each task
        if 'reg' in logits:
            registered = torch.from_numpy(logits['reg']['registered'])
            metrics = compute_metrics(registered, raw)
            for k, v in metrics.items():
                results['registration'][k].append(v)
        
        if 'fus' in logits:
            fused = torch.from_numpy(logits['fus']['fused'])
            metrics = compute_metrics(fused, raw)
            for k, v in metrics.items():
                results['fusion'][k].append(v)
        
        if 'SR' in logits:
            super_resolved = torch.from_numpy(logits['SR']['super_resolved'])
            metrics = compute_metrics(super_resolved, raw)
            for k, v in metrics.items():
                results['super_resolution'][k].append(v)
        
        if 'IR' in logits:
            restored = torch.from_numpy(logits['IR']['restored'])
            metrics = compute_metrics(restored, raw)
            for k, v in metrics.items():
                results['isotropic_restoration'][k].append(v)

print(f"\n✓ Inference complete!")
print(f"  Total samples: {total_samples}")
print(f"  Total batches: {len(inference_times)}")
print(f"  Total time: {sum(inference_times):.2f}s")
print(f"  Avg time/batch: {np.mean(inference_times):.4f}s")
print(f"  Avg time/sample: {sum(inference_times)/total_samples:.4f}s")
print(f"  Throughput: {total_samples/sum(inference_times):.2f} samples/s")

## 6. Results Analysis

In [ ]:
# Aggregate results
print("\n" + "="*80)
print("QUANTITATIVE RESULTS")
print("="*80)

results_summary = {}

for task_name, metrics_dict in results.items():
    if len(metrics_dict['mse']) > 0:
        print(f"\n{task_name.upper().replace('_', ' ')}:")
        print("-" * 80)
        
        task_summary = {}
        for metric_name, values in metrics_dict.items():
            mean_val = np.mean(values)
            std_val = np.std(values)
            min_val = np.min(values)
            max_val = np.max(values)
            
            task_summary[metric_name] = {
                'mean': float(mean_val),
                'std': float(std_val),
                'min': float(min_val),
                'max': float(max_val)
            }
            
            print(f"  {metric_name.upper():8s}: {mean_val:10.6f} ± {std_val:8.6f}  "
                  f"[{min_val:10.6f}, {max_val:10.6f}]")
        
        results_summary[task_name] = task_summary

print("\n" + "="*80)

## 7. Export Results

In [ ]:
# Create output directory
from datetime import datetime
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_dir = Path("test_results") / timestamp
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Saving results to: {output_dir}\n")

# 1. Save summary as JSON
summary_path = output_dir / "summary.json"
with open(summary_path, 'w') as f:
    json.dump({
        'checkpoint': str(CHECKPOINT_PATH),
        'config': {
            'img_size': config.img_size,
            'batch_size': TEST_BATCH_SIZE,
            'task': config.task,
            'embed_dim': config.embed_dim
        },
        'dataset': {
            'total_samples': total_samples,
            'num_batches': len(inference_times)
        },
        'timing': {
            'total_time_s': float(sum(inference_times)),
            'avg_time_per_batch_s': float(np.mean(inference_times)),
            'avg_time_per_sample_s': float(sum(inference_times)/total_samples),
            'throughput_samples_per_s': float(total_samples/sum(inference_times))
        },
        'metrics': results_summary
    }, f, indent=2)
print(f"  ✓ Saved: {summary_path.name}")

# 2. Save detailed results as CSV for each task
for task_name, metrics_dict in results.items():
    if len(metrics_dict['mse']) > 0:
        df = pd.DataFrame(metrics_dict)
        csv_path = output_dir / f"{task_name}_detailed.csv"
        df.to_csv(csv_path, index=False)
        print(f"  ✓ Saved: {csv_path.name}")

# 3. Save timing info
timing_df = pd.DataFrame({
    'batch_idx': range(len(inference_times)),
    'inference_time_s': inference_times
})
timing_path = output_dir / "timing.csv"
timing_df.to_csv(timing_path, index=False)
print(f"  ✓ Saved: {timing_path.name}")

print(f"\n✓ All results saved to: {output_dir}")

## 8. Performance Summary

In [ ]:
print("\n" + "="*80)
print("PERFORMANCE SUMMARY")
print("="*80)

print(f"\nModel:")
print(f"  Total parameters: {total_params/1e6:.2f}M")
print(f"  Trainable parameters: {trainable_params/1e6:.2f}M")

print(f"\nDataset:")
print(f"  Total samples evaluated: {total_samples}")
print(f"  Batch size: {TEST_BATCH_SIZE}")
print(f"  Number of batches: {len(inference_times)}")

print(f"\nTiming:")
print(f"  Total inference time: {sum(inference_times):.2f}s")
print(f"  Average time per batch: {np.mean(inference_times):.4f}s")
print(f"  Average time per sample: {sum(inference_times)/total_samples:.4f}s")
print(f"  Throughput: {total_samples/sum(inference_times):.2f} samples/s")

if torch.cuda.is_available():
    print(f"\nGPU Memory:")
    print(f"  Allocated: {torch.cuda.memory_allocated()/1024**3:.2f} GB")
    print(f"  Reserved: {torch.cuda.memory_reserved()/1024**3:.2f} GB")
    print(f"  Max allocated: {torch.cuda.max_memory_allocated()/1024**3:.2f} GB")

print("\nBest Performing Task (by PSNR):")
best_task = None
best_psnr = -float('inf')
for task_name, task_summary in results_summary.items():
    psnr = task_summary['psnr']['mean']
    if psnr > best_psnr:
        best_psnr = psnr
        best_task = task_name
print(f"  {best_task.replace('_', ' ').title()}: {best_psnr:.2f} dB")

print("\n" + "="*80)
print("✓ Testing complete!")
print("="*80)